In [1]:
import sys, os
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from arxiv_agent.observability import check_connection, get_client, flush

assert check_connection(), "Langfuse not reachable at localhost:3000 — is it running?"
print("Langfuse connection OK")
print(f"Langfuse UI: {os.environ['LANGFUSE_BASE_URL']}")

Langfuse connection OK
Langfuse UI: http://localhost:3000


In [2]:
from arxiv_agent.observability import start_trace, flush
from arxiv_agent.agent import run_agent

question = "What is the main contribution of the Attention Is All You Need paper?"

trace = start_trace(question)
result = run_agent(question, trace=trace)
flush()

print(f"\nAnswer:\n{result['answer']}")
print(f"\nLangfuse trace: {os.environ['LANGFUSE_BASE_URL']}/trace/{result['trace_id']}")

/Users/rajatgupta/repos/agent-observability-eval/arxiv-agent/.venv/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer



Answer:
Based on the paper **"Attention Is All You Need"** (arXiv:1706.03762), the main contribution is the proposal of a new network architecture called the **Transformer**. 

The key aspects of this contribution include:

*   **Reliance Solely on Attention:** The Transformer completely discards recurrence (RNNs/LSTMs) and convolutions. Instead, it relies entirely on attention mechanisms to draw global dependencies between input and output sequences.
*   **Introduction of Self-Attention:** The paper popularizes the use of **self-attention** (specifically multi-head attention) to allow the model to model relationships between all elements of a sequence simultaneously, regardless of their distance from one another.
*   **Massive Parallelization:** By removing the sequential nature of recurrent models, the Transformer allows for significantly more parallelization during training. This results in models that are much faster to train while requiring fewer GPU resources.
*   **State-of-the

In [3]:
from langfuse import Langfuse
import time

lf = get_client()
time.sleep(3)  # give Langfuse time to ingest the trace

raw = lf.api.trace.get(result["trace_id"])
print(f"Trace ID: {raw.id}")
print(f"Input: {raw.input}")
print(f"Output: {raw.output}")
print(f"\nSpans ({len(raw.observations)}):")
for obs in raw.observations:
    print(f"  [{obs.type}] {obs.name} — {obs.metadata}")

Trace ID: 47099ea552fee70056043e09908df0ee
Input: What is the main contribution of the Attention Is All You Need paper?
Output: Based on the paper **"Attention Is All You Need"** (arXiv:1706.03762), the main contribution is the proposal of a new network architecture called the **Transformer**. 

The key aspects of this contribution include:

*   **Reliance Solely on Attention:** The Transformer completely discards recurrence (RNNs/LSTMs) and convolutions. Instead, it relies entirely on attention mechanisms to draw global dependencies between input and output sequences.
*   **Introduction of Self-Attention:** The paper popularizes the use of **self-attention** (specifically multi-head attention) to allow the model to model relationships between all elements of a sequence simultaneously, regardless of their distance from one another.
*   **Massive Parallelization:** By removing the sequential nature of recurrent models, the Transformer allows for significantly more parallelization during

In [4]:
import json
from pathlib import Path
from arxiv_agent.agent import build_graph, SYSTEM_PROMPT
from langchain_core.messages import HumanMessage, SystemMessage

cases = json.loads(Path("../data/eval_cases.json").read_text())
results = []

for case in cases:
    print(f"Running {case['id']}: {case['question'][:60]}...")
    trace = start_trace(case["question"])

    graph = build_graph()
    msgs = graph.invoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=case["question"])],
        "trace": trace,
    })

    final_answer = msgs["messages"][-1].content
    trace.update(output=final_answer)
    trace.end()

    tool_calls_made = []
    for m in msgs["messages"]:
        if hasattr(m, "tool_calls") and m.tool_calls:
            for tc in m.tool_calls:
                tool_calls_made.append({"name": tc["name"], "args": tc["args"]})

    results.append({
        "id": case["id"],
        "category": case["category"],
        "question": case["question"],
        "answer": final_answer,
        "trace_id": trace.trace_id,
        "tool_calls_made": tool_calls_made,
    })
    flush()

Path("../data/run_results.json").write_text(json.dumps(results, indent=2))
print(f"\nSaved {len(results)} results to data/run_results.json")

Running kp_01: What is the main contribution of the Attention Is All You Ne...
Running kp_02: Summarize the RLHF paper by Christiano et al. 2017...
Running kp_03: What does the BERT paper propose?...
Running kp_04: Explain the LoRA fine-tuning method from paper 2106.09685...
Running sq_01: Find a paper on retrieval augmented generation and explain h...
Running sq_02: What recent work exists on chain of thought prompting?...
Running sq_03: Find papers on mixture of experts models...
Running nt_01: What is the difference between supervised and unsupervised l...
Running nt_02: What does gradient descent do in neural network training?...
Running bi_01: Tell me about arxiv paper 9999.99999...
Running bi_02: asdkjhasd lkjhaksjdh paper please...

Saved 11 results to data/run_results.json
